# Solving the head equation
In the following script we will solve the heat equation on complex geometry objects using finite element methods from DOLFINx. to accomplish this. We will first build a working minimal version in this jupyter notebook before translating the methods into a proper python script. We will aim to simulate a block of Tofu inside a cast iron skillet, heated from the bottom. Further we will try to generalize our code to such a degree, that we can load in objects to put in the pan.
## Arriving at the variational problem
To solve the heat equation we shall first take a look at the original PDE to solve. this is given by
$$
\begin{equation}
\begin{aligned}
\rho c \frac{\partial u}{\partial t} & = k  \nabla^2u + f  \qquad &\text{on } \Omega\times (0,T],\\
k \frac{\partial u}{\partial n} & = h \left(u-u_\text{env}\right)+\epsilon \sigma \left(u^4-u_\text{env}^4\right)\qquad &\text{on } \partial\Omega\times (0,T],\\
u & = u_0  \qquad &\text{at } t=0.
\end{aligned}
\end{equation}\tag{1}
$$
Here $u$ is a function of spatial coordinates $x$ and time $t$. The Domain $\Omega$ is the object on which the heat equation shall be solved, $n$ is the outward pointing normal vector on the domain, while $u_0$ is the function of the initial temperature distrbution at $t=0$. $f$ is a volume heat term, we will formally keep it in our equation for now but we need to remind ourselves that the pan does not have an integrated heater, so in the end $f=0$ for our pan. Yet there might be chemical reactions going on in the food that we cook so in general $f\neq 0$ inside that object. 
On the surface we include radiative and convection losses to the environment, where $u_\text{env}$ is the temperature of the environment. However whe should keep in mind, that this environment also includes the hot-plate at some point so we will split the surface of the domain in parts where it interacts with the hot-plate $\partial \Omega_\text{hp}$ and the free part $\partial \Omega_\text{free}$. Last but not least we shall shortly define all physical parameters:
$\rho$ is the density of the pan, $c$ is its heat coefficient, $k$ the thermal cunductivity, $h$ the convection transfer coefficient, $\epsilon$ the emissivity and $\sigma$ the Stefan-Boltzman constant. For ease of notation we define
$$\alpha:=\frac{k}{\rho c},\qquad \beta = \frac{h}{\rho c} \qquad \text{and} \qquad \gamma = \frac{\epsilon \sigma}{\rho c}$$
Further we shall scale $f \to f / (\rho c)$.

We will handle the time derivative by discretizing the problem into steps at points in time $t_n$. The PDE in discretized form can then be given by
$$\left(\frac{\partial u}{\partial t}\right)^n = \alpha \nabla^2 u^{n} + f^n,$$
where the index $n$ conveys that $u^n=u(x,t_n)$ is evaluated at a certain time. There is now the option to approximate the time derivative as the difference of $u$ between time steps. We write
$$\left(\frac{\partial u}{\partial t}\right)^{n+1}\approx\frac{u^{n+1}-u^n}{\Delta t}$$
and plug this into the expression above to obtain
$$\frac{u^{n+1}-u^n}{\Delta t} = \alpha\nabla^2 u^{n+1} + f^{n+1},$$
with its boundary condition of $u^0=u_0$. We reorder this equation and split in into a part only containing functions dependent on $u^{n+1}$ and a part with all remaining functions:
$$u^{n+1}-\alpha \Delta t \nabla^2 u^{n+1} - u^n - \Delta t f^{n+1}=0, \qquad \text{ for } n=1,2,3,\dots. \tag{2}$$
This makes it abbundandly clear, that we can solve this problem as iterative stationary problems. After solving it for a certain time point $t_n$ we can then obtain the solution for $t_{n+1}$ by plugging into the equation again.

At this point the variational form of these problems is easy to obtain by multiplying (2) by a test function $v$ and intergating over the domain $\Omega$
$$ \int_\Omega vu^{n+1}-\alpha \Delta t v \nabla^2 u^{n+1} - v u^n - \Delta t v f^{n+1} \mathrm{d} V=\text{const.}$$
We will now use integration by parts to obtain

$$\int_\Omega v u^{n+1}+ \alpha \Delta t (\nabla u^{n+1})\cdot(\nabla v) - v u^n - \Delta t v f^{n+1}\mathrm{d}V - \int_{\partial \Omega} \Delta t \alpha v n\cdot(\nabla u^{n+1}) \mathrm{d} S = \text{const.}$$
This can now be split up into a linear and bilinear part with

$$ \begin{equation}
\begin{aligned}
a(u^{n+1},v)&= \int_\Omega v u^{n+1} + \alpha \Delta t (\nabla u^{n+1}) \cdot (\nabla v) \mathrm{d} V\\
&-\int_{\partial \Omega} (\Delta t \beta) v u^{n+1} + (\Delta t \gamma) v (u^{n+1})^4 \mathrm{d}S\\
L_{n+1}(v)&=\int_\Omega v u^n + \Delta t v f^{n+1}\mathrm{d} V\\
&+\int_{\partial \Omega} (\Delta t \beta) v u_\text{env} + (\Delta t \gamma) v (u_\text{env})^4 \mathrm{d}S\\
\end{aligned}
\end{equation}$$
We now have to solve the variational problem 
$a(u^{n+1},v) = L_{n+1}(v) \quad \forall  v \in V$ with $V$ the vector space of functions on our domain. We notice that this problem is non linear on accord of the radiation term.

## Humble beginnings
We shall start with solving the heat-equation for a simple object, box with length $d$ which shall be heated from the bottom. We will for now ignore the radiation term ($\gamma =0$) and solve a simpler linear variation of the problem.

In [1]:
from dolfinx import default_scalar_type, plot
from dolfinx.fem import (
    Constant,
    Function,
    functionspace,
    assemble_scalar,
    dirichletbc,
    form,
    locate_dofs_topological,
    extract_function_spaces,
)
from dolfinx.fem.petsc import (
    assemble_vector,
    assemble_matrix,
    create_vector,
    apply_lifting,
    set_bc,
)
from dolfinx.io import XDMFFile
from dolfinx.mesh import create_box, locate_entities, meshtags, CellType
from dolfinx.plot import vtk_mesh
from pathlib import Path

from mpi4py import MPI
from ufl import (
    FacetNormal,
    Measure,
    SpatialCoordinate,
    TestFunction,
    TrialFunction,
    div,
    dot,
    dx,
    grad,
    inner,
    lhs,
    rhs,
)

from petsc4py import PETSc

import numpy as np
import pyvista
import sys
import matplotlib as mpl
import simulation

Now defining the mdomain by using a dolfinx box

In [2]:
d= 2

t=0.0
T=60.0
DeltaT=0.1
num_steps=int(T/DeltaT)

temp_dir = Path("temp/simulation_data")
temp_dir.mkdir(exist_ok=True,parents=True)
res_dir = Path("results")
res_dir.mkdir(exist_ok=True,parents=True)


class material:
    density = {
        "cast_iron" : 7.2,
        "tofu" : 1.05,
        "steel" : 7.85,
        "copper" : 8.9,
    }
    heat_capacity = {
        "cast_iron" : 0.5,
        "tofu" : 3.2,
        "steel" : 0.45,
        "copper" : 0.385
    }
    thermal_conductivity = {
        "cast_iron" : 0.4,
        "tofu" : 0.04,
        "steel" : 0.5,
        "copper" : 3.9
    }
    heat_transfer =  {
        "metal-metal" : 0.1,
        "metal_air" : 0.005,
        "tofu_air" : 0.001,
    }
    

skillet_material = "copper"
food_material = "copper"

beta_heat = 0.1
beta_amb = 0.001

T_ambient= 22
T_stove= 200

with XDMFFile(MPI.COMM_WORLD, "temp/meshing/sloped_skillet.xdmf", "r") as xdmf:
    domain = xdmf.read_mesh(name="Grid")
    cell_tags = xdmf.read_meshtags(domain, name="Grid")

    
V = functionspace(domain, ("Lagrange", 1))

C =  functionspace(domain, ("DG", 0))

skillet_cells = cell_tags.find(1)
food_cells  = cell_tags.find(2)
alpha = Function(C)
beta = Function(C)

alpha.x.array[skillet_cells] = material.thermal_conductivity[skillet_material]/(material.density[skillet_material]*material.heat_capacity[skillet_material])
alpha.x.array[food_cells] = material.thermal_conductivity[food_material]/(material.density[food_material]*material.heat_capacity[food_material])

x = SpatialCoordinate(domain)

u = TrialFunction(V)
v = TestFunction(V)

u_heat= Constant(domain,default_scalar_type(T_stove))
u_amb= Constant(domain,default_scalar_type(T_ambient))

#set initial conditions
def initial_condition(x):
    return np.full_like(x[0], T_ambient)

u_n = Function(V)
u_n.name = "u_n"
u_n.interpolate(initial_condition)

# define variational form
F = v*u*dx + alpha* DeltaT*inner(grad(u),grad(v))*dx-v*u_n*dx

We define functions to fix the boundary conditions

In [3]:
boundaries = [
    (1, lambda x: np.isclose(x[2], 0)),
    (2, lambda x: np.logical_not(np.isclose(x[2], 0)))
]

# Tagging faces for boundaries
facet_indices, facet_markers = [], []
fdim = domain.topology.dim - 1
for marker, locator in boundaries:
    facets = locate_entities(domain, fdim, locator)
    facet_indices.append(facets)
    facet_markers.append(np.full_like(facets, marker))
facet_indices = np.hstack(facet_indices).astype(np.int32)
facet_markers = np.hstack(facet_markers).astype(np.int32)
sorted_facets = np.argsort(facet_indices)
facet_tag = meshtags(
    domain, fdim, facet_indices[sorted_facets], facet_markers[sorted_facets]
)


debugging boundary conditions

In [4]:
domain.topology.create_connectivity(domain.topology.dim - 1, domain.topology.dim)
with XDMFFile(domain.comm, "temp/simulation_data/facet_tags.xdmf", "w") as xdmf:
    xdmf.write_mesh(domain)
    xdmf.write_meshtags(facet_tag, domain.geometry)

Define surface measure with tag 1 = heated and 2 = ambient

In [5]:
ds = Measure("ds", domain=domain, subdomain_data=facet_tag)

Define classes of boundary condtions. We will only need robin though

In [6]:
class Robin_BoundaryCondition:
    def __init__(self, marker, values):
        self._bc = values[0] * inner(u - values[1], v) * ds(marker)

    @property
    def bc(self):
        return self._bc

    @property
    def type(self):
        return self._type


# Define the boundary conditions
boundary_conditions = [
    Robin_BoundaryCondition(1, (beta_heat, u_heat)),
    Robin_BoundaryCondition(2, (beta_amb, u_amb)),
]

for condition in boundary_conditions:
    F += condition.bc

We shall design a time-dependent output. we will use xdmf to later visualize in paraview

In [7]:
with XDMFFile(domain.comm, "temp/simulation_data/diffusion.xdmf", "w") as xdmf:
    xdmf.write_mesh(domain)
    uh = Function(V)
    uh.name = "uh"
    uh.interpolate(initial_condition)

    xdmf.write_function(uh, t)

We now actually start solving the variational problem. We shall define

In [8]:
a = lhs(F)
L = rhs(F)

bilinear_form = form(a)
linear_form = form(L)

A = assemble_matrix(bilinear_form, bcs=[])
A.assemble()
b= create_vector(extract_function_spaces(linear_form))

In [9]:
# Solve linear variational problem
solver = PETSc.KSP().create(domain.comm)
solver.setOperators(A)
solver.setType(PETSc.KSP.Type.PREONLY)
solver.getPC().setType(PETSc.PC.Type.LU)

In [10]:
grid = pyvista.UnstructuredGrid(*plot.vtk_mesh(V))

plotter = pyvista.Plotter()
plotter.open_gif("results/u_time.gif", fps=int(1/DeltaT))

grid.point_data["uh"] = uh.x.array
warped = grid.warp_by_scalar("uh", factor=1)

viridis = mpl.colormaps.get_cmap("viridis").resampled(25)
sargs = dict(
    title_font_size=25,
    label_font_size=20,
    fmt="%.2e",
    color="black",
    position_x=0.1,
    position_y=0.8,
    width=0.8,
    height=0.1,
)


renderer = plotter.add_mesh(
    grid,
    show_edges=False,
    cmap=viridis,
    scalar_bar_args=sargs,
    clim=[0, T_stove],
)


In [11]:
simulation.run_simulation(t,T,DeltaT,uh,u_n,b,linear_form,solver,xdmf,plotter,A,grid)

|████████████████████████████████████████| 100.00%
Simulation Completed.

Writing Files...
Done!

Cleaning up...
Done!
